### 1. Contextualização

- Objetivo: Seu objetivo é prever se um cliente irá subscrever (aderir) a um depósito a prazo bancário.

- As submissões são avaliadas usando a AUC-ROC entre o valor previsto e o alvo observado.

### 2. Bibliotecas

In [1]:
import pandas as pd
import numpy as np

import warnings
import os

# Visualização
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

In [2]:
# Definir diretório
os.chdir("G:/Meu Drive/MeuDrive2/academico/3.kaggle/5.Binary_Classification_with_a_Bank_Dataset")

# Evitar avisos
warnings.filterwarnings('ignore')

# Exibir todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

### 3. Importando os dados

In [3]:
# Dados de treino
train = pd.read_csv("train.csv")

# Dados de teste
test = pd.read_csv("test.csv")

# Submissão
sample_submission = pd.read_csv("sample_submission.csv")

### 4. EDA

In [4]:
# Substituir 'unknown' por NaN em todo o DataFrame
train = train.replace("unknown", np.nan)
test = test.replace("unknown", np.nan)

In [5]:
# Dados faltantes por coluna
train.isna().mean()*100

id            0.000000
age           0.000000
job           0.388933
marital       0.000000
education     2.839867
default       0.000000
balance       0.000000
housing       0.000000
loan          0.000000
contact      30.883600
day           0.000000
month         0.000000
duration      0.000000
campaign      0.000000
pdays         0.000000
previous      0.000000
poutcome     89.660000
y             0.000000
dtype: float64

In [6]:
# Remover poutcome, pois há muitas variáveis faltantes, quase 90%
train = train.drop(columns=['poutcome', 'contact'])
test = test.drop(columns=['poutcome', 'contact'])

### 5. Separar em treino e validação

In [7]:
from sklearn.model_selection import train_test_split

# Separando em treino e teste
target = "y"
X = train.drop(columns=[target, 'id'])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.20, random_state=42, shuffle=True)

### 6. Dados faltantes

In [8]:
# Proporção de dados faltantes por coluna no conjunto de treino
X_train.isna().mean()*100

age          0.000000
job          0.387667
marital      0.000000
education    2.832000
default      0.000000
balance      0.000000
housing      0.000000
loan         0.000000
day          0.000000
month        0.000000
duration     0.000000
campaign     0.000000
pdays        0.000000
previous     0.000000
dtype: float64

In [9]:
# Proporção de dados faltantes por coluna no conjunto de teste
X_valid.isna().mean()*100

age          0.000000
job          0.394000
marital      0.000000
education    2.871333
default      0.000000
balance      0.000000
housing      0.000000
loan         0.000000
day          0.000000
month        0.000000
duration     0.000000
campaign     0.000000
pdays        0.000000
previous     0.000000
dtype: float64

In [10]:
# SimpleImputer
imputer = SimpleImputer(strategy='most_frequent')

# Ajustando aos dados de treino
imputer.fit(X_train)

SimpleImputer(strategy='most_frequent')

In [11]:
# Aplicando aos dados de treino e validação
X_train = imputer.transform(X_train)
X_valid = imputer.transform(X_valid)

In [12]:
# retornando para um df

# Treino
X_train = pd.DataFrame(
    X_train,
    columns=imputer.get_feature_names_out(),  # the variable names
)

# validação
X_valid = pd.DataFrame(
    X_valid,
    columns=imputer.get_feature_names_out(),  # the variable names
)

### 7. Criação de variáveis

In [13]:
X_train.head()

,age,job,marital,education,default,balance,housing,loan,day,month,duration,campaign,pdays,previous
0,28,blue-collar,single,secondary,no,5090,yes,yes,12,may,1297,2,-1,0
1,51,technician,married,tertiary,no,1295,no,no,27,aug,119,9,-1,0
2,57,management,divorced,tertiary,no,0,no,no,29,jan,87,1,-1,0
3,48,blue-collar,single,primary,no,1323,yes,no,15,may,83,5,-1,0
4,38,admin.,married,secondary,no,659,yes,no,28,jul,534,4,-1,0


#### 7.1 Dummies para pdays e previous

In [14]:
# Cria a dummy
X_train['pdays_is_minus1'] = (X_train['pdays'] == -1).astype(int)
X_valid['pdays_is_minus1'] = (X_valid['pdays'] == -1).astype(int)

In [15]:
# Cria a dummy
X_train['previous_is_zero'] = (X_train['previous'] == 0).astype(int)
X_valid['previous_is_zero'] = (X_valid['previous'] == 0).astype(int)

In [16]:
# Dropando as colunas
X_train = X_train.drop(['pdays', 'previous'], axis=1)
X_valid = X_valid.drop(['pdays', 'previous'], axis=1)

#### 7.2 Ordinal Encoding

Utilizando esse tipo de encoding por ser mais adequado a modelos baseados em árvores

In [17]:
# Selecionando os atributos 
cat_vars = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'month']
print(cat_vars)

['job', 'marital', 'education', 'default', 'housing', 'loan', 'month']


In [18]:
# Ajustando o encoder
encoder = OrdinalEncoder()

In [19]:
# Ajustando o transformer as colunas

ct = ColumnTransformer(
    [("oe", encoder, cat_vars)],
    remainder="passthrough",
)

ct.set_output(transform="pandas")

ColumnTransformer(remainder='passthrough',
                  transformers=[('oe', OrdinalEncoder(),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'month'])])

In [20]:
# "Aprendendo" nos dados de treino
ct.fit(X_train)

ColumnTransformer(remainder='passthrough',
                  transformers=[('oe', OrdinalEncoder(),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'month'])])

In [21]:
# Aplicando ao treino e ao teste
X_train_enc = ct.transform(X_train)
X_valid_enc = ct.transform(X_valid)

In [22]:
X_train_enc.head()

,oe__job,oe__marital,oe__education,oe__default,oe__housing,oe__loan,oe__month,remainder__age,remainder__balance,remainder__day,remainder__duration,remainder__campaign,remainder__pdays_is_minus1,remainder__previous_is_zero
0,1.0,2.0,1.0,0.0,1.0,1.0,8.0,28,5090,12,1297,2,1,1
1,9.0,1.0,2.0,0.0,0.0,0.0,1.0,51,1295,27,119,9,1,1
2,4.0,0.0,2.0,0.0,0.0,0.0,4.0,57,0,29,87,1,1,1
3,1.0,2.0,0.0,0.0,1.0,0.0,8.0,48,1323,15,83,5,1,1
4,0.0,1.0,1.0,0.0,1.0,0.0,5.0,38,659,28,534,4,1,1


### 8. Padronizando os dados

In [23]:
from sklearn.preprocessing import MinMaxScaler

# Instancia o scaler
scaler = MinMaxScaler()

# Aprende apenas no conjunto de treino
scaler.fit(X_train_enc)

# Aplica no treino, validação e teste
X_train_scaled = scaler.transform(X_train_enc)
X_valid_scaled = scaler.transform(X_valid_enc)

# Voltando a estrutura df
X_train_scaled = pd.DataFrame(
    X_train_scaled, 
    columns=X_train_enc.columns,  # mantém os nomes originais
    index=X_train_enc.index       # mantém os índices
)

X_valid_scaled = pd.DataFrame(
    X_valid_scaled, 
    columns=X_valid_enc.columns, 
    index=X_valid_enc.index
)

### 9. Predição

Para aplicar SMOTE junto com GridSearchCV, você precisa usar um pipeline, porque o SMOTE deve ser aplicado somente no conjunto de treinamento dentro de cada fold da validação cruzada, e não antes de tudo (senão você estaria vazando informações).

#### 9.2 XGBoost

In [24]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score
from scipy.stats import randint, uniform

# Pipeline com SMOTE + XGBoost
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('xgb', XGBClassifier(
        random_state=42,
        use_label_encoder=False,
        eval_metric="auc"
    ))
])

# Distribuições ampliadas dos hiperparâmetros
param_dist = {
    "xgb__n_estimators": randint(100, 2001),       # de 100 a 2000 árvores
    "xgb__max_depth": randint(3, 16),              # de 3 a 15
    "xgb__learning_rate": uniform(0.01, 0.29),    # de 0.01 a 0.3
    "xgb__subsample": uniform(0.5, 0.5)           # de 0.5 a 1.0
}

# RandomizedSearch
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=50,  
    scoring=make_scorer(roc_auc_score, needs_proba=True),
    cv=3,
    verbose=3,
    random_state=42
)

# Treina
random_search.fit(X_train_scaled, y_train)

print("Melhores parâmetros:", random_search.best_params_)
print("Melhor ROC AUC:", random_search.best_score_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV 1/3] END xgb__learning_rate=0.11861663446573512, xgb__max_depth=15, xgb__n_estimators=1394, xgb__subsample=0.8659969709057025;, score=nan total time= 5.2min
[CV 2/3] END xgb__learning_rate=0.11861663446573512, xgb__max_depth=15, xgb__n_estimators=1394, xgb__subsample=0.8659969709057025;, score=nan total time= 5.3min
[CV 3/3] END xgb__learning_rate=0.11861663446573512, xgb__max_depth=15, xgb__n_estimators=1394, xgb__subsample=0.8659969709057025;, score=nan total time= 2.7min
[CV 1/3] END xgb__learning_rate=0.18361096041714062, xgb__max_depth=9, xgb__n_estimators=221, xgb__subsample=0.5779972601681014;, score=nan total time=  24.7s
[CV 2/3] END xgb__learning_rate=0.18361096041714062, xgb__max_depth=9, xgb__n_estimators=221, xgb__subsample=0.5779972601681014;, score=nan total time=  24.6s
[CV 3/3] END xgb__learning_rate=0.18361096041714062, xgb__max_depth=9, xgb__n_estimators=221, xgb__subsample=0.5779972601681014;, score=n

Melhores parâmetros: {'xgb__learning_rate': 0.11861663446573512, 'xgb__max_depth': 15, 'xgb__n_estimators': 1394, 'xgb__subsample': 0.8659969709057025}
Melhor ROC AUC: nan

### 10. Teste

In [25]:
# Removendo a coluna id do X_test (mas vamos precisar dela para o submission)
ids = test["id"].copy()
X_test = test.drop(columns=["id"])

# Dados faltantes
X_test = imputer.transform(X_test)
X_test = pd.DataFrame(
    X_test,
    columns=imputer.get_feature_names_out()
)

In [26]:
# Criando dummies para pdays e previous
X_test["pdays_is_minus1"] = (X_test["pdays"] == -1).astype(int)
X_test["previous_is_zero"] = (X_test["previous"] == 0).astype(int)

# Dropando as colunas originais
X_test = X_test.drop(["pdays", "previous"], axis=1)

In [27]:
# Ordinal encoding
X_test_enc = ct.transform(X_test)

In [28]:
# Padronização
X_test_scaled = scaler.transform(X_test_enc)

# Voltando para DataFrame
X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test_enc.columns,
    index=X_test_enc.index
)

In [29]:
# Usamos predict_proba porque a competição espera probabilidades
test_preds_proba = random_search.predict_proba(X_test_scaled)[:, 1]

### Submissão

In [30]:
# Criar submissão no formato exigido
submission = sample_submission.copy()
submission["y"] = test_preds_proba  # substitui a coluna y

In [31]:
# Garantir que o id esteja correto
submission["id"] = ids.values

In [32]:
# Salvar em CSV
submission.to_csv("sub_final.csv", index=False)